## Reading data

In [ ]:
import numpy as np
import polars as pl
import glob
from enum import Enum

class CountMode(Enum):
    TOTAL = 0
    TWOBYONE = 1
    THREEBYONE = 2

def unrank(ranks, base=6, dims=12):
    ranks = np.array(ranks, dtype=np.int64)
    powers = base ** np.arange(dims-1,-1,-1)
    unranked = (ranks[:, None] // powers) % base
    return unranked

    
# lazy_df = pl.scan_csv("part*.txt")

def _finalize_if_lazy(df_or_lf: pl.DataFrame | pl.LazyFrame) -> pl.DataFrame:
    if isinstance(df_or_lf, pl.LazyFrame):
        return (
            df_or_lf
            .group_by(pl.all())
            .agg(pl.len())
            .collect(streaming=True)
        )
    return df_or_lf

def _calculate_piece_count(df_or_lf: pl.DataFrame | pl.LazyFrame, mode : CountMode) -> pl.Expr:
    base = 6
    dims = 12
    dim_exprs = []

    if isinstance(df_or_lf, pl.LazyFrame):
        cluster_col = pl.col("Cluster").str.replace("0x", "").str.to_integer(base=16)
    else:
        cluster_col = pl.col("Cluster")

    for i in range(dims):
        power = base ** (dims - 1 - i)
        dim_expr = (cluster_col // power) % base
        dim_exprs.append(dim_expr)

    count_1 = pl.sum_horizontal([expr == 1 for expr in dim_exprs])
    count_2 = pl.sum_horizontal([expr == 2 for expr in dim_exprs])
    count_3 = pl.sum_horizontal([expr == 3 for expr in dim_exprs])
    count_4 = pl.sum_horizontal([expr == 4 for expr in dim_exprs])
    count_5 = pl.sum_horizontal([expr == 5 for expr in dim_exprs])
   
    p_3x1 = count_1 + count_2 + count_3
    p_2x1 = count_2 + count_3 + count_4 + (2*count_5)

    if mode == CountMode.THREEBYONE:
        return p_3x1
    elif mode == CountMode.TWOBYONE:
        return p_2x1
    else:
        return p_2x1 + p_3x1
    
def _calculate_exact_piece_count(cluster_col : pl.Expr):
    base = 6
    dims = 12
    dim_exprs = []

    for i in range(dims):
        power = base ** (dims - 1 - i)
        dim_expr = (cluster_col // power) % base
        dim_exprs.append(dim_expr)

    count_1 = pl.sum_horizontal([expr == 1 for expr in dim_exprs])
    count_2 = pl.sum_horizontal([expr == 2 for expr in dim_exprs])
    count_3 = pl.sum_horizontal([expr == 3 for expr in dim_exprs])
    count_4 = pl.sum_horizontal([expr == 4 for expr in dim_exprs])
    count_5 = pl.sum_horizontal([expr == 5 for expr in dim_exprs])
   
    p_3x1 = count_1 + count_2 + count_3
    p_2x1 = count_2 + count_3 + count_4 + (2*count_5)

    return p_2x1, p_3x1

def cluster_filter_by_size(df_or_lf : pl.DataFrame | pl.LazyFrame, min : int, max : int) -> pl.DataFrame:
    result = df_or_lf.filter(pl.col("Cluster Size").is_between(min, max))
    return _finalize_if_lazy(result)

def cluster_filter_by_length(df_or_lf : pl.DataFrame | pl.LazyFrame, min : int, max : int) -> pl.DataFrame:
    result = df_or_lf.filter(pl.col("Max Length").is_between(min, max))
    return _finalize_if_lazy(result)

def cluster_filter_by_pieces(df_or_lf : pl.DataFrame | pl.LazyFrame, mode : CountMode, target_count : int) -> pl.DataFrame:
    if isinstance(df_or_lf, pl.LazyFrame):
        prepared_df = df_or_lf.with_columns(
            Cluster=pl.col("Cluster").str.replace("0x", "").str.to_integer(base=16)
        )
        piece_count_expr = _calculate_piece_count(df_or_lf, mode)
    
        query = (
            prepared_df
            .filter(piece_count_expr == target_count)
            .group_by(pl.all())
            .agg(pl.len())
        )
        df_result = query.collect(streaming=True)

        df_result = df_result.with_columns(
            Cluster=pl.col("Cluster").map_elements(lambda x: hex(x), return_dtype=pl.String)
        )
        return df_result
    else:
        if df_or_lf.schema["Cluster"] == pl.String:
            working_df = df_or_lf.with_columns(
                Cluster=pl.col("Cluster").str.replace("0x", "").str.to_integer(base=16)
            )
        else:
            working_df = df_or_lf
        piece_count_expr = _calculate_piece_count(pl.col("Cluster"), mode)
        df_result = working_df.filter(piece_count_expr == target_count)
        df_result = df_result.with_columns(
            Cluster=pl.col("Cluster").map_elements(lambda x: hex(x), return_dtype=pl.String)
        )
        return df_result

def cluster_filter_by_exact_pieces(df_or_lf : pl.DataFrame | pl.LazyFrame, target_2x1: int, target_3x1: int) -> pl.DataFrame:
    if isinstance(df_or_lf, pl.LazyFrame):
        prepared_df = df_or_lf.with_columns(
            Cluster=pl.col("Cluster").str.replace("0x", "").str.to_integer(base=16)
        )

        p_2x1, p_3x1 = _calculate_exact_piece_count(pl.col("Cluster"))

        query = (
            prepared_df
            .filter((p_2x1 == target_2x1) & (p_3x1 == target_3x1))
            .group_by(pl.all())
            .agg(pl.len())
        )
        df_result = query.collect(streaming=True)

        df_result = df_result.with_columns(
           Cluster=pl.col("Cluster").map_elements(lambda x: hex(x), return_dtype=pl.String)
        )
        return df_result
    else:
        if df_or_lf.schema["Cluster"] == pl.String:
            working_df = df_or_lf.with_columns(
                Cluster=pl.col("Cluster").str.replace("0x", "").str.to_integer(base=16)
            )
        else:
            working_df = df_or_lf
        
        p_2x1, p_3x1 = _calculate_exact_piece_count(pl.col("Cluster"))

        df_result = working_df.filter((p_2x1==target_2x1) & (p_3x1==target_3x1))
        df_result = df_result.with_columns(
            Cluster=pl.col("Cluster").map_elements(lambda x: hex(x), return_dtype=pl.String)
        )
        return df_result
#     base = 6
#     dims = 12
#     dim_exprs = []

#     for i in range(dims):
#         power = base ** (dims - 1 - i)
#         dim_expr = (pl.col("Cluster") // power) % base
#         dim_exprs.append(dim_expr)

#     count_1 = pl.sum_horizontal([expr == 1 for expr in dim_exprs])
#     count_2 = pl.sum_horizontal([expr == 2 for expr in dim_exprs])
#     count_3 = pl.sum_horizontal([expr == 3 for expr in dim_exprs])
#     count_4 = pl.sum_horizontal([expr == 4 for expr in dim_exprs])
#     count_5 = pl.sum_horizontal([expr == 5 for expr in dim_exprs])

#     p_3x1 = count_1 + count_2 + count_3
#     p_2x1 = count_2 + count_3 + count_4 + (2 * count_5)

#     query = (
#         lazy_df
#         .with_columns(Cluster=pl.col("Cluster").str.replace("0x", "").str.to_integer(base=16))
#         .filter((p_2x1 == target_2x1) & (p_3x1 == target_3x1))
        
#         .group_by(pl.all())
#         .agg(pl.len())
#     )

#     df_result = query.collect(streaming=True)
#     df_result = df_result.with_columns(
#         Cluster=pl.col("Cluster").map_elements(lambda x: hex(x), return_dtype=pl.String)
#     )
#     return df_result
    
# def transform_clusters_for_input(df : pl.Series):
#     base = 6
#     if df.dtype == pl.String:
#         df = df.str.replace("0x", "").str.to_integer(base=16)
#     ranks = df.to_numpy()
#     powers = base ** np.arange(11, -1, -1, dtype=np.uint64)
#     matrix = (ranks[:, None] // powers) % base
#     matrix = np.delete(matrix, 8, axis=1)
#     new_powers = base ** np.arange(10, -1, -1, dtype=np.uint64)
#     new_ranks = np.dot(matrix, new_powers)
#     new_ranks = new_ranks.astype(int).tolist()
#     new_ranks = [ hex(int(i)) for i in new_ranks]
#     return new_ranks

In [7]:
lazy_df = pl.scan_csv("part*.txt")
df = cluster_filter_by_length(lazy_df, 10, 15)
df = cluster_filter_by_pieces(df, CountMode.TOTAL, 11)
# df = cluster_filter_by_size(df, 200000, 600000)
print(df)

/tmp/ipykernel_6208/836796812.py:26: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


shape: (231_947, 7)
┌────────────┬───────┬──────────────┬──────────────┬────────────┬───────┬─────┐
│ Cluster    ┆ Legal ┆ Has Solution ┆ Cluster Size ┆ Max Length ┆ Time  ┆ len │
│ ---        ┆ ---   ┆ ---          ┆ ---          ┆ ---        ┆ ---   ┆ --- │
│ str        ┆ i64   ┆ i64          ┆ i64          ┆ i64        ┆ f64   ┆ u32 │
╞════════════╪═══════╪══════════════╪══════════════╪════════════╪═══════╪═════╡
│ 0x1130d4c  ┆ 1     ┆ 1            ┆ 2729         ┆ 14         ┆ 0.073 ┆ 1   │
│ 0x1132174  ┆ 1     ┆ 1            ┆ 3811         ┆ 14         ┆ 0.045 ┆ 1   │
│ 0x19d1fad  ┆ 1     ┆ 1            ┆ 3704         ┆ 13         ┆ 0.046 ┆ 1   │
│ 0x19d7a81  ┆ 1     ┆ 1            ┆ 537          ┆ 10         ┆ 0.018 ┆ 1   │
│ 0x19df431  ┆ 1     ┆ 1            ┆ 210          ┆ 11         ┆ 0.006 ┆ 1   │
│ …          ┆ …     ┆ …            ┆ …            ┆ …          ┆ …     ┆ …   │
│ 0x6562443e ┆ 1     ┆ 1            ┆ 5729         ┆ 11         ┆ 0.094 ┆ 1   │
│ 0x6562716c ┆ 1    

In [4]:
df = cluster_filter_by_length(1, 10)
print(df)

/tmp/ipykernel_2137/407598287.py:36: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_result = query.collect(streaming=True)


shape: (14_517_319, 7)
┌──────────┬───────┬──────────────┬──────────────┬────────────┬───────┬─────┐
│ Cluster  ┆ Legal ┆ Has Solution ┆ Cluster Size ┆ Max Length ┆ Time  ┆ len │
│ ---      ┆ ---   ┆ ---          ┆ ---          ┆ ---        ┆ ---   ┆ --- │
│ str      ┆ i64   ┆ i64          ┆ i64          ┆ i64        ┆ f64   ┆ u32 │
╞══════════╪═══════╪══════════════╪══════════════╪════════════╪═══════╪═════╡
│ 0x1ca5fa ┆ 1     ┆ 1            ┆ 657          ┆ 1          ┆ 0.003 ┆ 1   │
│ 0x1cb52e ┆ 1     ┆ 1            ┆ 876          ┆ 7          ┆ 0.005 ┆ 1   │
│ 0x1cb54b ┆ 1     ┆ 1            ┆ 1872         ┆ 8          ┆ 0.031 ┆ 1   │
│ 0x1cbf34 ┆ 1     ┆ 1            ┆ 1332         ┆ 4          ┆ 0.004 ┆ 1   │
│ 0x1cca03 ┆ 1     ┆ 1            ┆ 2247         ┆ 9          ┆ 0.025 ┆ 1   │
│ …        ┆ …     ┆ …            ┆ …            ┆ …          ┆ …     ┆ …   │
│ 0x631c25 ┆ 1     ┆ 1            ┆ 604          ┆ 1          ┆ 0.004 ┆ 1   │
│ 0x8315e6 ┆ 1     ┆ 1            ┆ 2359 

In [5]:
print(df['Cluster Size'].sum())

18910038679


In [13]:
df = cluster_filter_by_size(200000, 600000)
print(df)

/tmp/ipykernel_548637/1294640368.py:13: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_result = query.collect(streaming=True)


shape: (298, 7)
┌────────────┬───────┬──────────────┬──────────────┬────────────┬───────┬─────┐
│ Cluster    ┆ Legal ┆ Has Solution ┆ Cluster Size ┆ Max Length ┆ Time  ┆ len │
│ ---        ┆ ---   ┆ ---          ┆ ---          ┆ ---        ┆ ---   ┆ --- │
│ str        ┆ i64   ┆ i64          ┆ i64          ┆ i64        ┆ f64   ┆ u32 │
╞════════════╪═══════╪══════════════╪══════════════╪════════════╪═══════╪═════╡
│ 0x6562d20c ┆ 1     ┆ 1            ┆ 262010       ┆ 10         ┆ 5.234 ┆ 1   │
│ 0x6752500c ┆ 1     ┆ 1            ┆ 246890       ┆ 6          ┆ 2.502 ┆ 1   │
│ 0x656499ac ┆ 1     ┆ 1            ┆ 270068       ┆ 8          ┆ 3.804 ┆ 1   │
│ 0x6564e08c ┆ 1     ┆ 1            ┆ 271925       ┆ 6          ┆ 2.949 ┆ 1   │
│ 0x675417a9 ┆ 1     ┆ 1            ┆ 200770       ┆ 5          ┆ 0.243 ┆ 1   │
│ …          ┆ …     ┆ …            ┆ …            ┆ …          ┆ …     ┆ …   │
│ 0x26ea6164 ┆ 1     ┆ 1            ┆ 200770       ┆ 6          ┆ 1.849 ┆ 1   │
│ 0x67cc0cb0 ┆ 1     ┆ 1

In [55]:
df = cluster_filter_by_pieces(CountMode.TOTAL, 2)
print(df)

/tmp/ipykernel_548637/1464831295.py:73: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_result = query.collect(streaming=True)


shape: (22, 7)
┌────────────┬───────┬──────────────┬──────────────┬────────────┬──────┬─────┐
│ Cluster    ┆ Legal ┆ Has Solution ┆ Cluster Size ┆ Max Length ┆ Time ┆ len │
│ ---        ┆ ---   ┆ ---          ┆ ---          ┆ ---        ┆ ---  ┆ --- │
│ str        ┆ i64   ┆ i64          ┆ i64          ┆ i64        ┆ f64  ┆ u32 │
╞════════════╪═══════╪══════════════╪══════════════╪════════════╪══════╪═════╡
│ 0x361      ┆ 1     ┆ 1            ┆ 20           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x159fdb60 ┆ 1     ┆ 1            ┆ 17           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x364      ┆ 1     ┆ 1            ┆ 25           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x366      ┆ 1     ┆ 1            ┆ 20           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x99c960   ┆ 1     ┆ 1            ┆ 14           ┆ 2          ┆ 0.0  ┆ 1   │
│ …          ┆ …     ┆ …            ┆ …            ┆ …          ┆ …    ┆ …   │
│ 0xe6a9360  ┆ 1     ┆ 1            ┆ 21           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x567f6360 ┆ 1     ┆ 1            ┆

In [58]:
df = cluster_filter_by_exact_pieces(1, 1)
print(df)

/tmp/ipykernel_548637/1267056461.py:107: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_result = query.collect(streaming=True)


shape: (11, 7)
┌───────────┬───────┬──────────────┬──────────────┬────────────┬──────┬─────┐
│ Cluster   ┆ Legal ┆ Has Solution ┆ Cluster Size ┆ Max Length ┆ Time ┆ len │
│ ---       ┆ ---   ┆ ---          ┆ ---          ┆ ---        ┆ ---  ┆ --- │
│ str       ┆ i64   ┆ i64          ┆ i64          ┆ i64        ┆ f64  ┆ u32 │
╞═══════════╪═══════╪══════════════╪══════════════╪════════════╪══════╪═════╡
│ 0x366     ┆ 1     ┆ 1            ┆ 20           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x21c0    ┆ 1     ┆ 1            ┆ 20           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x99c960  ┆ 1     ┆ 1            ┆ 14           ┆ 2          ┆ 0.0  ┆ 1   │
│ 0x19a460  ┆ 1     ┆ 1            ┆ 14           ┆ 2          ┆ 0.0  ┆ 1   │
│ 0xb9a0    ┆ 1     ┆ 1            ┆ 17           ┆ 2          ┆ 0.0  ┆ 1   │
│ …         ┆ …     ┆ …            ┆ …            ┆ …          ┆ …    ┆ …   │
│ 0x870     ┆ 1     ┆ 1            ┆ 20           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x448e0   ┆ 1     ┆ 1            ┆ 14          

In [103]:
import subprocess
import os
list = transform_clusters_for_input(df["Cluster"])

with open("temp.txt", "w") as file:
    for cluster in list:
        print(cluster, file=file)

binary_path = "./retrograde_analysis/main -f temp.txt -b 1 -a retro -m length > output_boards_length.txt"
result = subprocess.run(binary_path, shell=True)

binary_path = "./retrograde_analysis/main -f temp.txt -b 1 -a retro -m entropy > output_boards_entropy.txt"
result = subprocess.run(binary_path, shell=True)

binary_path = "./retrograde_analysis/main -f temp.txt -b 1 -a retro -m e-l > output_boards_e-l.txt"
result = subprocess.run(binary_path, shell=True)

os.remove("temp.txt")